# Summarizer — service eval notebook

Hits the deployed `/summarize` endpoint for all four profiles:

| Profile | Input fields | Output |
|---------|-------------|--------|
| `article` | `title`, `text` | `headline`, `summary` |
| `entity_desc` | `name`, `type`, `subtype?`, `evidence[]` | `description` |
| `relation_desc` | `head`, `tail`, `type`, `subtype?`, `attributes`, `evidence[]` | `description` |
| `aggregate` | `name`, `type`, `descriptions[]`, `evidence[]` | `description` |

**Structural checks** (automated):
- `article`: headline word count 8–15, summary sentence count 2–4 (same rules as the service validator)
- All profiles: output non-empty, under the expected sentence cap
- Extractive reduction: flagged when input exceeds 400 words (`SUMMARIZE_MAX_INPUT_WORDS` default)

**Quality metrics** (ROUGE): every case has a hand-written reference output; ROUGE-1/2/L F1 are computed against it.

Test cases live in `eval/fixtures/summarize-news-summarizer.json`.

In [ ]:
import os, json, re, requests
from pathlib import Path

NLP_BASE_URL = os.environ.get('NLP_BASE_URL', 'http://localhost:9907').rstrip('/')
ENDPOINT     = f'{NLP_BASE_URL}/summarize'
HEADERS      = {'Content-Type': 'application/json', 'X-API-Key': f'{os.environ.get("NLP_API_KEY")}'}
TIMEOUT      = 120

MAX_INPUT_WORDS = 400   # mirrors SUMMARIZE_MAX_INPUT_WORDS default in the service

CASES = json.loads(Path('fixtures/summarize-news-summarizer.json').read_text(encoding='utf-8'))
print(f'{len(CASES)} cases -> {ENDPOINT}')

In [ ]:
def call_summarize(inp):
    r = requests.post(ENDPOINT, json=inp, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json().get('result', {})


_SENTENCE_END = re.compile(r'[.!?]+')

def count_words(text):
    return len(text.split())

def count_sentences(text):
    parts = _SENTENCE_END.split(text)
    return sum(1 for p in parts if p.strip())

def extractive_would_trigger(fields):
    """Return (triggered, word_count) for the designated long field."""
    text = fields.get('text') or ''
    evidence = fields.get('evidence', [])
    if isinstance(evidence, list):
        evidence_text = ' '.join(evidence)
    else:
        evidence_text = str(evidence)
    long_field = text if text else evidence_text
    wc = count_words(long_field)
    return wc > MAX_INPUT_WORDS, wc

In [ ]:
def render(result, case):
    profile  = case['input']['profile']
    fields   = case['input']['fields']
    checks   = case.get('expected', {}).get('checks', {})

    triggered, wc = extractive_would_trigger(fields)
    ext_note = f'  ⚡ extractive reduction triggered ({wc} words > {MAX_INPUT_WORDS})' if triggered \
               else f'  (extractive not triggered: {wc} words)'
    print(ext_note)

    if profile == 'article':
        headline = result.get('headline', '')
        summary  = result.get('summary',  '')
        hw = count_words(headline)
        ss = count_sentences(summary)
        lo_h, hi_h = checks.get('headline_words', [8, 15])
        lo_s, hi_s = checks.get('summary_sentences', [2, 4])
        h_ok = lo_h <= hw <= hi_h
        s_ok = lo_s <= ss <= hi_s
        print(f'  HEADLINE ({hw} words) {"✅" if h_ok else "❌"}')
        print(f'    {headline!r}')
        print(f'  SUMMARY ({ss} sentences) {"✅" if s_ok else "❌"}')
        print(f'    {summary!r}')
        return h_ok and s_ok

    else:
        description = result.get('description', '')
        sc = count_sentences(description)
        max_s = checks.get('max_sentences', 5)
        non_empty = bool(description.strip())
        under_cap = sc <= max_s
        print(f'  DESCRIPTION ({sc} sentences) {"✅" if non_empty and under_cap else "❌"}')
        print(f'    {description!r}')
        return non_empty and under_cap

In [ ]:
results = []   # (case_id, passed, result_dict)
for case in CASES:
    profile = case['input']['profile']
    print(f"── {case['id']}: {case['description']}  [{profile}]")
    try:
        result = call_summarize(case['input'])
    except Exception as exc:
        print('   ERROR:', exc); print()
        results.append((case['id'], False, {}))
        continue

    checks = case.get('expected', {}).get('checks', {})
    passed = render(result, case)

    if checks.get('extractive_triggered'):
        triggered, _ = extractive_would_trigger(case['input']['fields'])
        if not triggered:
            print('  ⚠️  expected extractive reduction but input is under the word budget')

    if checks.get('no_source_suffix') and profile == 'article':
        headline = result.get('headline', '')
        if re.search(r'[-|]\s*\S+\.\S+', headline):
            print(f'  ⚠️  headline may still contain source suffix: {headline!r}')
            passed = False

    icon = '✅' if passed else '❌'
    print(f'   {icon} {"pass" if passed else "fail"}')
    print()
    results.append((case['id'], passed, result))

In [ ]:
passed_n = sum(1 for _, ok, _ in results if ok)
print('SCORECARD · /summarize')
print(f'  Cases passing: {passed_n}/{len(results)}')
for cid, ok, _ in results:
    print(f"   {'✅' if ok else '❌'} {cid}")

## ROUGE quality metrics

Each case has a hand-written reference output. ROUGE-1/2/L F1 measures lexical overlap between the generated text and the reference.

- **ROUGE-1**: unigram overlap — content coverage
- **ROUGE-2**: bigram overlap — fluency / phrasing similarity  
- **ROUGE-L**: longest common subsequence — structural similarity

Scores are F1 (harmonic mean of precision and recall). For abstractive summarisation, ROUGE-1 > 0.4 is strong; ROUGE-2 > 0.15 is acceptable.

> Install if needed: `pip install rouge-score`

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

case_map  = {c['id']: c for c in CASES}
result_map = {cid: res for cid, _, res in results}

rouge_rows = []

for cid, passed, result in results:
    case      = case_map[cid]
    profile   = case['input']['profile']
    reference = case.get('expected', {}).get('reference', {})

    if not reference or not result:
        rouge_rows.append((cid, profile, None, None, None, '—'))
        continue

    if profile == 'article':
        # score headline and summary separately, report the summary (more informative)
        h_scores = scorer.score(reference.get('headline',''), result.get('headline',''))
        s_scores = scorer.score(reference.get('summary',''),  result.get('summary',''))
        r1 = s_scores['rouge1'].fmeasure
        r2 = s_scores['rouge2'].fmeasure
        rl = s_scores['rougeL'].fmeasure
        h_r1 = h_scores['rouge1'].fmeasure
        note = f'headline R1={h_r1:.2f}'
    else:
        s_scores = scorer.score(reference.get('description',''), result.get('description',''))
        r1 = s_scores['rouge1'].fmeasure
        r2 = s_scores['rouge2'].fmeasure
        rl = s_scores['rougeL'].fmeasure
        note = ''

    rouge_rows.append((cid, profile, r1, r2, rl, note))

print(f"{'ID':<10} {'Profile':<14} {'R-1':>6} {'R-2':>6} {'R-L':>6}  Notes")
print('-' * 60)
for cid, profile, r1, r2, rl, note in rouge_rows:
    if r1 is None:
        print(f'{cid:<10} {profile:<14}  (no result or reference)')
    else:
        flag = '⚠️ ' if r1 < 0.25 else ('✅' if r1 >= 0.40 else '')
        print(f'{cid:<10} {profile:<14} {r1:>6.2f} {r2:>6.2f} {rl:>6.2f}  {flag} {note}'.rstrip())

scored = [(r1,r2,rl) for _,_,r1,r2,rl,_ in rouge_rows if r1 is not None]
if scored:
    print('-' * 60)
    print(f"{'mean':<24} {sum(r for r,_,_ in scored)/len(scored):>6.2f} "
          f"{sum(r for _,r,_ in scored)/len(scored):>6.2f} "
          f"{sum(r for _,_,r in scored)/len(scored):>6.2f}")